# langfuse trace review with reactcode and label studio

This notebook is the companion for the **Langfuse → Label Studio Enterprise** ReactCode integration tutorial.

You will:
- pull traces + observations from Langfuse
- normalize them into a consistent trace-review schema
- create a Label Studio project via API
- import traces as tasks
- review + annotate in a custom ReactCode UI

> **Requirement:** ReactCode is available in **Label Studio Enterprise** only.


## 1) Configuration

Create a `.env` file in the repository root (or the same directory as this notebook) with the following variables:

```bash
# Label Studio Enterprise
LABEL_STUDIO_HOST=http://localhost:8080       # or your LS Enterprise instance URL
LABEL_STUDIO_API_KEY=your_label_studio_api_key

# Langfuse
LANGFUSE_BASE_URL=https://cloud.langfuse.com  # or your self-hosted Langfuse URL
LANGFUSE_PUBLIC_KEY=your_langfuse_public_key
LANGFUSE_SECRET_KEY=your_langfuse_secret_key
LANGFUSE_PROJECT=your_project_name            # project name (for display in Label Studio)

# OpenAI (only needed for Section 3a sample trace generation)
OPENAI_API_KEY=your_openai_api_key
```

> **Note:** Langfuse API keys (public + secret key pair) are already scoped to a specific project, so `LANGFUSE_PROJECT` is used as a display name only — no project ID resolution needed.


In [14]:
import os
from dotenv import load_dotenv

# Load .env from current directory or repository root
# override=True ensures kernel picks up .env changes without restart
load_dotenv(override=True)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'), override=True)  # fallback: repo root

# Label Studio Enterprise
LABEL_STUDIO_HOST = os.getenv('LABEL_STUDIO_HOST', 'http://localhost:8080')
LABEL_STUDIO_API_KEY = os.getenv('LABEL_STUDIO_API_KEY', '')

# Langfuse
LANGFUSE_BASE_URL = os.getenv('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', '')
LANGFUSE_PROJECT = os.getenv('LANGFUSE_PROJECT', '')

# OpenAI (only needed for sample trace generation in Section 3a)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

print('LABEL_STUDIO_HOST:', LABEL_STUDIO_HOST)
print('LANGFUSE_BASE_URL:', LANGFUSE_BASE_URL)
print('LANGFUSE_PROJECT:', LANGFUSE_PROJECT or '(not set — will fetch all traces)')
print('Has LABEL_STUDIO_API_KEY?', bool(LABEL_STUDIO_API_KEY))
print('Has LANGFUSE keys?', bool(LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY))
print('Has OPENAI_API_KEY?', bool(OPENAI_API_KEY))

LABEL_STUDIO_HOST: https://app.humansignal.com
LANGFUSE_BASE_URL: https://cloud.langfuse.com
LANGFUSE_PROJECT: ls-project
Has LABEL_STUDIO_API_KEY? True
Has LANGFUSE keys? True
Has OPENAI_API_KEY? True


## 2) install dependencies


In [8]:
!pip -q install requests label-studio-sdk python-dotenv langfuse langchain langchain-openai langgraph



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 3) label studio reactcode config
3-panel UI (turn list → detail → annotation) for trace review. Loaded from `label_config.py` in this directory.


In [15]:
# Load 3-panel ReactCode config (label_config.py + template.js in same directory)
_config_paths = [
  'label_config.py',
  os.path.join(os.getcwd(), 'tutorials', 'how-to-review-langfuse-traces-with-label-studio', 'label_config.py'),
]
_config_path = next((p for p in _config_paths if os.path.exists(p)), None)
if _config_path:
  import importlib.util
  _spec = importlib.util.spec_from_file_location('label_config', _config_path)
  _mod = importlib.util.module_from_spec(_spec)
  _spec.loader.exec_module(_mod)
  LABEL_CONFIG_XML = _mod.LABEL_CONFIG_XML
else:
  raise FileNotFoundError('label_config.py not found. Ensure it is in the same directory as this notebook.')
print(LABEL_CONFIG_XML[:300] + '\n...')

<View>
  <ReactCode style="height: 95vh" name="trace" toName="trace" outputs='{"trace_id":"string","turn_id":"string","turn_role":"string","verdict":"string","failure_modes":"array","severity":"string","expected_behavior":"string","comments":"string"}'>
    <![CDATA[
    function TraceAnnotator({ Re
...


## 3a) Generate sample traces (optional)

If you already have traces in Langfuse, **skip this cell** — set `GENERATE_TRACES = False` and go directly to Section 4.

Otherwise, this cell creates a ReAct agent with multiple tools and runs 4 multi-turn conversations to produce realistic traces in your Langfuse project. Requires `OPENAI_API_KEY`.

In [10]:
GENERATE_TRACES = True  # Set to False if you already have traces in Langfuse

if GENERATE_TRACES:
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI
    from langchain.agents import create_agent
    from langfuse.langchain import CallbackHandler
    from langfuse import get_client

    if not OPENAI_API_KEY:
        raise RuntimeError('OPENAI_API_KEY is required for trace generation. Set it in your .env or set GENERATE_TRACES=False.')

    langfuse = get_client()

    # --- Tools ---
    @tool
    def calculator(expression: str) -> str:
        """Evaluate a math expression. Input should be a valid Python math expression."""
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

    @tool
    def search_knowledge_base(query: str) -> str:
        """Search an internal knowledge base for information about company policies, products, or procedures."""
        kb = {
            "refund": "Refund policy: Full refund within 30 days of purchase. After 30 days, store credit only. Damaged items: full refund at any time with photo evidence.",
            "shipping": "Shipping: Standard (5-7 business days, free over $50), Express (2-3 days, $12.99), Overnight ($24.99). International shipping available to 40+ countries.",
            "warranty": "Warranty: 1-year limited warranty on all electronics. 2-year extended warranty available for $29.99. Covers manufacturing defects only.",
            "pricing": "Enterprise pricing: Base plan $99/mo (up to 10 users), Pro plan $249/mo (up to 50 users), Enterprise plan custom pricing. Annual billing saves 20%.",
        }
        query_lower = query.lower()
        results = [v for k, v in kb.items() if k in query_lower]
        return results[0] if results else f"No results found for: {query}"

    @tool
    def get_weather(city: str) -> str:
        """Get the current weather for a city."""
        weather_data = {
            "new york": "New York: 72°F, Partly Cloudy, Humidity 65%, Wind 8 mph SW",
            "london": "London: 58°F, Overcast, Humidity 80%, Wind 12 mph W",
            "tokyo": "Tokyo: 82°F, Clear, Humidity 55%, Wind 5 mph NE",
            "paris": "Paris: 63°F, Light Rain, Humidity 75%, Wind 10 mph NW",
        }
        return weather_data.get(city.lower(), f"Weather data not available for {city}")

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    agent = create_agent(llm, [calculator, search_knowledge_base, get_weather])

    # --- 4 multi-turn conversations ---
    # Each conversation uses start_as_current_observation() to group
    # all LangChain invocations under a single Langfuse trace.
    conversations = [
        # 1: Customer support with knowledge base lookups
        [
            "What's the refund policy for a product I bought 3 weeks ago?",
            "What if the item arrived damaged? Does that change anything?",
        ],
        # 2: Multi-step calculation with follow-up
        [
            "I need to calculate the total cost for our team: 35 users on the Pro plan with annual billing. What's the yearly cost?",
            "How much would we save compared to monthly billing?",
        ],
        # 3: Weather + planning across cities
        [
            "I'm planning a trip. What's the weather like in Tokyo and Paris right now?",
            "Based on the weather, which city would be better for outdoor sightseeing today?",
        ],
        # 4: Mixed tool usage — knowledge base + calculator
        [
            "What shipping options do you have, and how much would express shipping cost for 5 orders?",
            "If I spend $250 total on products, which shipping is free? And what's the total with express for the remaining orders?",
        ],
    ]

    for i, conv_messages in enumerate(conversations, 1):
        print(f"\n--- Conversation {i} ---")

        # Context manager groups all nested invocations under one trace
        with langfuse.start_as_current_observation(
            as_type="span",
            name=f"conversation_{i}",
            input={"messages": conv_messages},
        ) as root_span:
            handler = CallbackHandler()
            chat_history = []
            final_reply = ""

            for msg_text in conv_messages:
                print(f"  User: {msg_text[:80]}...")
                chat_history.append(HumanMessage(content=msg_text))

                result = agent.invoke(
                    {'messages': chat_history},
                    config={'callbacks': [handler]},
                )
                chat_history = result['messages']
                final_reply = result['messages'][-1].content
                print(f"  Assistant: {final_reply[:100]}...")

            root_span.update_trace(
                input={"messages": conv_messages},
                output={"response": final_reply},
            )

    langfuse.flush()
    print(f'\n✓ Generated {len(conversations)} traces (1 per conversation). Proceed to Section 4.')
else:
    print('Skipped trace generation. Proceed to Section 4.')



--- Conversation 1 ---
  User: What's the refund policy for a product I bought 3 weeks ago?...
  Assistant: The refund policy states that you can receive a full refund within 30 days of purchase. Since you bo...
  User: What if the item arrived damaged? Does that change anything?...
  Assistant: If the item arrived damaged, you can still receive a full refund at any time, as long as you provide...

--- Conversation 2 ---
  User: I need to calculate the total cost for our team: 35 users on the Pro plan with a...
  Assistant: The yearly cost for 35 users on the Pro plan with annual billing is $2,390.40....
  User: How much would we save compared to monthly billing?...
  Assistant: You would save approximately $597.60 compared to monthly billing....

--- Conversation 3 ---
  User: I'm planning a trip. What's the weather like in Tokyo and Paris right now?...
  Assistant: The current weather is as follows:

- **Tokyo**: 82°F, Clear, Humidity 55%, Wind 5 mph NE
- **Paris*...
  User: Based o

## 4) Langfuse API client

Fetches traces and observations from the Langfuse REST API. Your API keys are already scoped to a specific Langfuse project, so all traces returned belong to that project.


In [16]:
import base64
from typing import Any, Dict, List, Optional

import requests


def _basic_auth(public_key: str, secret_key: str) -> str:
  token = base64.b64encode(f'{public_key}:{secret_key}'.encode('utf-8')).decode('utf-8')
  return f'Basic {token}'


class LangfuseClient:
  def __init__(self, base_url: str, public_key: str, secret_key: str):
    self.base_url = base_url.rstrip('/')
    self.s = requests.Session()
    self.s.headers.update({
      'Authorization': _basic_auth(public_key, secret_key),
      'Content-Type': 'application/json'
    })

  def list_traces(self, limit: int = 20, page: int = 1, from_ts: Optional[str] = None, to_ts: Optional[str] = None) -> Dict[str, Any]:
    url = f'{self.base_url}/api/public/traces'
    params: Dict[str, Any] = { 'limit': limit, 'page': page, 'fields': 'core' }
    if from_ts:
      params['fromTimestamp'] = from_ts
    if to_ts:
      params['toTimestamp'] = to_ts
    r = self.s.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

  def get_trace(self, trace_id: str) -> Dict[str, Any]:
    url = f'{self.base_url}/api/public/traces/{trace_id}'
    r = self.s.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

  def list_observations_v2(self, trace_id: str, limit: int = 200) -> List[Dict[str, Any]]:
    """Fetch observations for a trace via v1 API (avoids v2 parseIoAsJson 400)."""
    url = f'{self.base_url}/api/public/observations'
    out: List[Dict[str, Any]] = []
    page = 1
    page_size = min(max(limit, 1), 100)
    while True:
      params: Dict[str, Any] = {
        'traceId': trace_id,
        'page': page,
        'limit': page_size,
      }
      r = self.s.get(url, params=params, timeout=60)
      r.raise_for_status()
      payload = r.json()
      data = payload.get('data') or []
      out.extend(data)
      if len(data) < page_size:
        break
      page += 1
    return out


lf = LangfuseClient(LANGFUSE_BASE_URL, LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY)
print('Langfuse client ready')


Langfuse client ready


## 5) Normalize Langfuse traces → unified schema

Converts Langfuse traces + observations into a flat list of turns (user, assistant, tool) that the ReactCode template can render. Each turn has a `role`, `content` (always a string), and optional `tool_calls`, `usage`, and `model`.


In [17]:
import json as _json

def _to_str(x):
    """Safely convert any value to a readable string."""
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    if isinstance(x, (dict, list)):
        try:
            return _json.dumps(x, indent=2, default=str)
        except Exception:
            return str(x)
    return str(x)


def _extract_content(obj):
    """Extract plain-text content from various Langfuse data shapes."""
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        # {"content": "..."} or {"text": "..."} etc.
        for key in ('content', 'text', 'input', 'output', 'result'):
            if isinstance(obj.get(key), str) and obj[key].strip():
                return obj[key]
        return _to_str(obj)
    if isinstance(obj, list):
        parts = [_extract_content(item) for item in obj if _extract_content(item).strip()]
        return '\n'.join(parts) if parts else _to_str(obj)
    return str(obj)


def normalize_langfuse_trace(trace, observations):
    """Convert a Langfuse trace + observations into the unified schema.

    LangGraph produces this observation tree:
      conversation (SPAN, root)
        └── LangGraph (CHAIN)
              ├── agent (AGENT)
              │     ├── call_model (CHAIN)
              │     │     └── ChatOpenAI (GENERATION)  ← LLM input/output
              │     └── should_continue (CHAIN)
              ├── tools (CHAIN)
              │     └── search_knowledge_base (TOOL)   ← tool execution
              ├── agent (AGENT)  ← next reasoning step
              ...

    Strategy:
      1. Walk observations chronologically
      2. GENERATION → extract user messages from input + assistant response from output
      3. TOOL → extract tool execution as a tool turn
      4. Skip structural nodes (CHAIN, AGENT, SPAN) that are just wrappers
    """
    trace_id = trace.get('id') or trace.get('traceId')
    session_id = trace.get('sessionId') or trace_id

    obs_sorted = sorted(observations, key=lambda o: o.get('startTime') or o.get('createdAt') or '')

    turns = []
    turn_counter = 0
    seen_user_messages = set()

    def add_turn(role, content, **kwargs):
        nonlocal turn_counter
        if not content or not content.strip():
            return
        turn = {
            'turn_id': f'turn_{turn_counter}',
            'role': role,
            'content': content.strip(),
            'timestamp': kwargs.get('timestamp', ''),
        }
        for k in ('model', 'usage', 'tool_calls', 'tool_name', 'tool_input'):
            if k in kwargs and kwargs[k]:
                turn[k] = kwargs[k]
        turns.append(turn)
        turn_counter += 1

    for obs in obs_sorted:
        otype = (obs.get('type') or '').upper()
        ts = obs.get('startTime') or obs.get('createdAt') or ''
        inp = obs.get('input')
        out = obs.get('output')

        # ── GENERATION: LLM call with messages in, response out ──
        if otype == 'GENERATION':
            # Extract user messages from input (list of role/content dicts)
            if isinstance(inp, list):
                for msg in inp:
                    if isinstance(msg, dict) and msg.get('role') == 'user':
                        content = msg.get('content', '')
                        if isinstance(content, list):
                            content = ' '.join(
                                p.get('text', '') if isinstance(p, dict) else str(p) for p in content
                            )
                        if content and content.strip():
                            msg_key = content[:200]
                            if msg_key not in seen_user_messages:
                                seen_user_messages.add(msg_key)
                                add_turn('user', content, timestamp=ts)

            # Extract assistant response from output
            if isinstance(out, dict):
                assistant_content = out.get('content', '')
                tool_calls_raw = out.get('tool_calls', [])

                # Build tool_calls metadata for the assistant turn
                tool_calls = []
                for tc in tool_calls_raw:
                    if isinstance(tc, dict):
                        tool_calls.append({
                            'tool_name': tc.get('name', 'unknown'),
                            'input': _to_str(tc.get('args', tc.get('input', ''))),
                            'call_id': tc.get('id', ''),
                        })

                # If assistant has content (actual response text)
                if assistant_content and assistant_content.strip():
                    add_turn(
                        'assistant', assistant_content,
                        timestamp=ts,
                        model=obs.get('model') or obs.get('providedModelName'),
                        usage=_normalize_usage(obs),
                        tool_calls=tool_calls if tool_calls else None,
                    )
                # If assistant has no content but made tool calls (thinking step)
                elif tool_calls:
                    tool_names = ', '.join(tc['tool_name'] for tc in tool_calls)
                    add_turn(
                        'assistant', f'[Calling: {tool_names}]',
                        timestamp=ts,
                        model=obs.get('model') or obs.get('providedModelName'),
                        usage=_normalize_usage(obs),
                        tool_calls=tool_calls,
                    )

        # ── TOOL: actual tool execution ──
        elif otype == 'TOOL':
            tool_name = obs.get('name') or 'unknown'
            tool_input = _to_str(inp) if inp else ''
            tool_output = _extract_content(out) if out else ''

            if tool_output:
                add_turn(
                    'tool', tool_output,
                    timestamp=ts,
                    tool_name=tool_name,
                    tool_input=tool_input,
                )

    # Fallback: if no turns extracted, use trace-level input/output
    if not turns:
        trace_input = _extract_content(trace.get('input'))
        trace_output = _extract_content(trace.get('output'))
        if trace_input:
            add_turn('user', trace_input, timestamp=trace.get('timestamp') or '')
        if trace_output:
            add_turn('assistant', trace_output, timestamp=trace.get('timestamp') or '')

    return {
        'trace_id': str(trace_id),
        'session_id': str(session_id),
        'metadata': {
            'name': trace.get('name'),
            'source': 'langfuse',
            'tags': trace.get('tags') or [],
            'start_time': trace.get('timestamp') or trace.get('createdAt') or '',
        },
        'turns': turns,
    }


def _normalize_usage(obs):
    """Extract token usage from an observation."""
    raw = obs.get('usageDetails') or obs.get('usage')
    if not isinstance(raw, dict):
        return None
    return {
        'input_tokens': raw.get('inputTokens') or raw.get('input_tokens') or raw.get('input') or 0,
        'output_tokens': raw.get('outputTokens') or raw.get('output_tokens') or raw.get('output') or 0,
    }


print('✓ Normalization functions defined')


✓ Normalization functions defined


## 6) Fetch, normalize, and import into Label Studio

Fetches traces from Langfuse, normalizes them, creates a Label Studio project with the ReactCode config, and imports the tasks.


In [18]:
from label_studio_sdk import LabelStudio
from label_studio_sdk.core.request_options import RequestOptions
from typing import Any, Dict, List

# Extended timeout for large ReactCode label configs
_REQUEST_OPTS = RequestOptions(timeout_in_seconds=120)


def create_project(ls_host: str, api_key: str, title: str, label_config: str) -> int:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    project = client.projects.create(title=title, label_config=label_config, request_options=_REQUEST_OPTS)
    return int(project.id)


def import_tasks(ls_host: str, api_key: str, project_id: int, tasks: List[Dict[str, Any]]) -> Any:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    return client.projects.import_tasks(id=project_id, request=tasks, return_task_ids=True)


if not LABEL_STUDIO_API_KEY:
    raise RuntimeError('Missing LABEL_STUDIO_API_KEY — set it in your .env file.')

# 1) Fetch traces from Langfuse
traces_payload = lf.list_traces(limit=20, page=1)
traces_list = traces_payload.get('data') or traces_payload.get('traces') or []

if not traces_list:
    raise RuntimeError(
        'No traces returned from Langfuse. Ensure you have traces in your project '
        'and that LANGFUSE_* credentials are correct. Run Section 3a to generate sample traces.'
    )

print(f'Fetched {len(traces_list)} traces from Langfuse')

# 2) Normalize each trace
tasks: List[Dict[str, Any]] = []
for t in traces_list:
    tid = t.get('id') or t.get('traceId')
    if not tid:
        continue
    full_trace = lf.get_trace(str(tid))
    obs = lf.list_observations_v2(str(tid))
    normalized = normalize_langfuse_trace(full_trace, obs)

    # Only include traces that have actual turns
    if normalized['turns']:
        tasks.append({'data': normalized})
        print(f"  + Trace {tid[:12]}... -> {len(normalized['turns'])} turns "
              f"({sum(1 for t in normalized['turns'] if t['role']=='user')} user, "
              f"{sum(1 for t in normalized['turns'] if t['role']=='assistant')} assistant, "
              f"{sum(1 for t in normalized['turns'] if t['role']=='tool')} tool)")

print(f'\nPrepared {len(tasks)} tasks for import')

# 3) Create project and import
project_id = create_project(
    ls_host=LABEL_STUDIO_HOST,
    api_key=LABEL_STUDIO_API_KEY,
    title=f'Langfuse Trace Review ({LANGFUSE_PROJECT})',
    label_config=LABEL_CONFIG_XML,
)
print(f'Created project: {project_id}')

resp = import_tasks(LABEL_STUDIO_HOST, LABEL_STUDIO_API_KEY, project_id, tasks)
print(f'Imported {len(tasks)} tasks')

print(f'\nDone! Open your project: {LABEL_STUDIO_HOST.rstrip("/")}/projects/{project_id}')

Fetched 4 traces from Langfuse
  + Trace d18b318bdaa6... -> 6 turns (2 user, 3 assistant, 1 tool)
  + Trace 7e0a5485ebc5... -> 7 turns (2 user, 3 assistant, 2 tool)
  + Trace c8deaec9b103... -> 10 turns (2 user, 5 assistant, 3 tool)
  + Trace 874755f93ce4... -> 6 turns (2 user, 3 assistant, 1 tool)

Prepared 4 tasks for import
Created project: 236879
Imported 4 tasks

Done! Open your project: https://app.humansignal.com/projects/236879


## What's next

- **Start annotating**: Open the project link above and click through traces in the ReactCode UI
- **Share with SMEs**: Invite domain experts to your Label Studio project for collaborative evaluation
- **Incremental sync**: Re-run sections 4–6 periodically to pull new traces
- **Custom taxonomy**: Edit `template.js` to add failure modes specific to your domain
- **Braintrust / LangSmith**: See companion tutorials for other observability platforms
